In [ ]:
import rubin_nights.dayobs_utils as rn_dayobs
day_obs = str(rn_dayobs.day_obs_str_to_int(rn_dayobs.today_day_obs()))
delta_hours = "2"

In [ ]:
# This forces an upgrade of the necessary packages, pip installing into the user's space
import getpass
if "noteburst" in getpass.getuser():
    print("Running in times square")
    print("Updating ts_fbs_utils")
    # Note that a regular user running this in jupyter lab can run these commands to get the packages pip installed as well
    !pip install --user --upgrade git+https://github.com/lsst-ts/ts_fbs_utils.git  --no-deps --no-build-isolation > /dev/null 2>&1

In [ ]:
import rubin_scheduler
import lsst.ts.fbs.utils
# check which versions available
print("rubin_scheduler", rubin_scheduler.__version__)
print("ts_fbs_utils", lsst.ts.fbs.utils.__version__)

In [ ]:
# RUBIN_SIM_DATA_DIR at usdf
import os
current_location = os.getenv("EXTERNAL_INSTANCE_URL", "")
if 'usdf' in current_location:
    os.environ["RUBIN_SIM_DATA_DIR"] = "/sdf/data/rubin/shared/rubin_sim_data"

In [ ]:
import healpy as hp
import numpy as np
from astropy.coordinates import SkyCoord, AltAz
from astropy.time import Time, TimeDelta
import astropy.units as u
from matplotlib.patches import Circle

from lsst.ts.fbs.utils.maintel.lsst_surveys import safety_masks
from rubin_scheduler.scheduler.features import Conditions
from rubin_scheduler.scheduler.model_observatory import ModelObservatory
from rubin_scheduler.utils import angular_separation, approx_ra_dec2_alt_az

import rubin_nights.dayobs_utils as rn_dayobs

def healpix_radec_to_altaz(in_hpx_map, nside=int(512/2)):
    npix = hp.nside2npix(nside)
    hpx_map = hp.ud_grade(in_hpx_map, nside)

    theta_altaz, phi_altaz = hp.pix2ang(nside, np.arange(npix))
    altaz_coords = SkyCoord(
        alt=(0.5*np.pi - theta_altaz) * u.rad,
        az=phi_altaz * u.rad,
        frame=AltAz(
            obstime=Time(conditions.mjd, format='mjd'),
            location=rn_dayobs.rubin_observer().location
        )
    )
    radec = altaz_coords.transform_to("icrs")

    pix = hp.ang2pix(
        nside,
        theta=(0.5*np.pi - radec.dec.to(u.rad).value),
        phi=radec.ra.to(u.rad).value
    )
    altaz_map = hpx_map[pix]

    return altaz_map

In [ ]:
# Convert times square parameters
day_obs = int(day_obs)
td = TimeDelta(float(delta_hours) * u.hour, format='sec')

sunset, sunrise = rn_dayobs.day_obs_sunset_sunrise(day_obs, sun_alt=-12)
times = np.arange(sunset.utc.mjd, sunrise.utc.mjd, td.jd)

In [ ]:
nside = 64
default_safety_masks = safety_masks(nside=nside,
                                    wind_speed_maximum=None,
                                    shadow_minutes=0, 
                                    apply_time_limited_shadow=False)

In [ ]:
model_observatory = ModelObservatory(nside=nside, 
                                     mjd=times[0],
                                     init_load_length=1)

In [ ]:
# from astropy.coordinates import EarthLocation, get_body, AltAz
# from astropy.time import Time
# import astropy.units as u

# loc = EarthLocation.of_site('rubin')

# obs_time = Time.now()
# moon_gcrs = get_body("moon", time=obs_time, location=loc)
# moon_altaz = moon_gcrs.transform_to(AltAz(obstime=obs_time, location=loc))

# print(f"Altitude: {moon_altaz.alt:.2f}")
# print(f"Azimuth: {moon_altaz.az:.2f}")


## The effect of the FBS standard masks (primarily moon avoidance) and an estimate of the galactic plane location.

When running AOS stability tests, or other tests being run from the FBS, the effect of masking 30 degrees around the moon can be limiting.
This notebook is attempting to help identify good locations on the sky (or times of night) for things like the AOS stability tests. 

The plots show alt-az views of the night sky, on DayObs {{ params.day_obs }}, at spacings of {{ params.delta_hours }}. 

The FBS standard masks are configured and shown, as well as an estimate of the location of the galactic plane. 
The white regions will not be scheduled with the FBS if the masks are in the standard configuration. 
The +/- 20 degrees galactic latitude is indicated in the lighter blue region. 
The best regions to observe would then be the darker blue areas. 



In [ ]:
moon_info = []
for time in times:
    model_observatory.mjd = time
    conditions = model_observatory.return_conditions()
    
    masks = np.zeros(hp.nside2npix(nside))
    for bf in default_safety_masks:
        masks += bf(conditions)
    
    sky = SkyCoord(ra = conditions.ra * u.rad, dec = conditions.dec * u.rad)
    gal_lat = sky.galactic.b.deg
    gal_alpha = np.where(np.abs(gal_lat) > 20, 1, 0.5)

    conditions_time = Time(conditions.mjd, format='mjd')
    m = pd.Series([conditions_time.utc.iso, np.degrees(conditions.moon_alt), np.degrees(conditions.moon_az), np.degrees(conditions.moon_ra), np.degrees(conditions.moon_dec), np.degrees(conditions.sun_alt)],
                  index=["Time (UTC)", "Moon Alt (deg)", "Moon Az (deg)", "Moon RA (deg)" , "Moon Dec (deg)", "Sun Alt (deg)"])
    moon_info.append(m)
    
    altaz_map = healpix_radec_to_altaz(masks)
    altaz_galalpha = healpix_radec_to_altaz(gal_alpha)
    
    rot=(180, 90, 0)
    alpha=np.ones(len(altaz_map)) * 0.5
    alpha = altaz_galalpha
    hp.azeqview(altaz_map, alpha=alpha, rot=rot, min=-1, max=1, cmap='tab10',
                title=f"LSST FBS masks at UTC {conditions_time.utc.iso}", cbar=False, badcolor='white', half_sky=True, lamb=True, flip="geo",
                reso=13/2, xsize=800*2, ysize=800*2)
    # Add a circle patch
    ax = plt.gca()
    circle = Circle((0.5, 0.5), radius=0.468, transform=ax.transAxes, 
                    fill=False, edgecolor='black', linewidth=1)
    ax.add_patch(circle)
    hp.graticule(dpar=20, dmer=45)
    plt.figtext(0.10, 0.5, "East (90)", fontsize='large')
    _ = plt.figtext(0.80, 0.5, "West (270)", fontsize='large')

moon_info = pd.DataFrame(moon_info)
moon_info = moon_info.set_index("Time (UTC)")
display(moon_info.round(0))